# 第十课：AI系统架构与持续进化 —— 从原型到生产

## 学习目标
- 理解护栏（Guardrails）在生产级五层架构中的位置
>
> 完整的五层架构（增强上下文 / 护栏 / 路由网关 / 缓存 / 智能体模式）属于概念轨，见教程文档课程十。
- 实现简单的输入护栏和输出过滤
- 设计用户反馈收集机制
- 完整回顾十节课的核心知识脉络

> 把前面九节课学到的东西串起来，理解一个真正的 AI 系统是如何运作的。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：为 AI 系统设计「护栏」

### 活动目标
护栏（Guardrails）是 AI 系统的安全防线。包括输入护栏（过滤有害输入）和输出护栏（过滤不当输出）。
这个活动让你用代码实现简单的护栏机制。

In [ ]:
# 活动一：实现输入护栏

# 定义敏感词列表
sensitive_keywords = ['密码', '银行卡号', '身份证号', '家庭住址', '裸照', '暴力', '自杀']

def input_guardrail(user_input):
    """输入护栏：检测并拦截不当输入"""
    hits = []
    for keyword in sensitive_keywords:
        if keyword in user_input:
            hits.append(keyword)
    if hits:
        return False, f'输入包含敏感词: {", ".join(hits)}，已拦截。'
    return True, '安全'

# 测试输入护栏
test_inputs = [
    '今天的天气怎么样？',
    '请帮我查一下我的银行卡号是多少？',
    '推荐一本关于AI的书给我',
    '如何用暴力解决问题？',
]

print('=== 输入护栏测试 ===')
for inp in test_inputs:
    ok, msg = input_guardrail(inp)
    status = '通过' if ok else '拦截'
    print(f'[{status}] {inp}')
    if not ok:
        print(f'  原因: {msg}')

### 输出护栏

In [ ]:
# 输出护栏：过滤 AI 的不当输出

def output_guardrail(ai_response):
    """输出护栏：检测AI回答中的风险内容"""
    warnings = []
    # 检查是否包含免责声明
    if '投资建议' in ai_response and '风险' not in ai_response:
        warnings.append('缺少风险提示')
    # 检查是否在「扮演医生」
    medical_keywords = ['诊断', '处方', '用药建议', '症状', '服用']
    for kw in medical_keywords:
        if kw in ai_response:
            warnings.append(f'涉及医疗建议({kw})，需要标注「仅供参考，请咨询医生」')
    return warnings

# 测试输出护栏
print('=== 输出护栏测试 ===')

# 模拟一个可能有问题的 AI 回答
risky_response = '根据你的症状，我建议你服用阿司匹林，每天两次。'
print(f'AI回答: {risky_response}')
warnings = output_guardrail(risky_response)
if warnings:
    print('警告：')
    for w in warnings:
        print(f'  - {w}')
else:
    print('输出检测通过，无风险。')

# 模拟一个安全的回答
safe_response = '关于你的健康问题，建议咨询专业医生。以下信息仅供参考：...'
print(f'\nAI回答: {safe_response}')
warnings = output_guardrail(safe_response)
if warnings:
    print('警告：')
    for w in warnings:
        print(f'  - {w}')
else:
    print('输出检测通过，无风险。')

### 讨论
- 输入护栏和输出护栏哪个更难实现？为什么？
- 护栏太严会限制 AI 能力，太松会有安全风险——如何平衡？
- 如果用户用隐喻或绕弯方式绕过护栏，怎么办？

---

## 活动二：设计用户反馈收集机制

### 活动目标
用户反馈是 AI 持续进化的燃料。设计一个好的反馈收集机制，是 AI 系统从「能用」到「好用」的关键。
反馈不是让你给用户发问卷——而是在对话流中自然地收集信号。

In [ ]:
# 活动二：模拟反馈收集系统

# 模拟一个对话历史
conversation = [
    {'role': 'user', 'content': '请问我的订单什么时候发货？'},
    {'role': 'assistant', 'content': '您的订单 #12345 预计明天发货，物流单号将通过短信发送给您。'},
    {'role': 'user', 'content': '好的谢谢！'},
    {'role': 'assistant', 'content': '不客气！还有其他可以帮您的吗？'},
]

# 反馈收集：在对话结束后自动分析
def analyze_conversation_feedback(conversation):
    """从对话中自动提取反馈信号"""
    signals = {
        'user_said_thanks': False,
        'user_asked_followup': False,
        'conversation_length': len(conversation),
        'resolution_likely': False
    }

    for idx, msg in enumerate(conversation):
        if msg['role'] == 'user':
            if any(w in msg['content'] for w in ['谢谢', '感谢', '太好了', '解决了']):
                signals['user_said_thanks'] = True
                signals['resolution_likely'] = True
            # 首轮提问不是「追问」：只有 AI 已经答过一次之后再提问才算
            if idx > 0 and ('?' in msg['content'] or '？' in msg['content']):
                signals['user_asked_followup'] = True

    return signals

signals = analyze_conversation_feedback(conversation)

print('=== 自动反馈分析 ===')
print(f'对话轮次: {signals["conversation_length"]}')
print(f'用户表达感谢: {signals["user_said_thanks"]}')
print(f'用户追问: {signals["user_asked_followup"]}')
print(f'可能已解决: {signals["resolution_likely"]}')

# 反馈总结
print('=== 反馈总结 ===')
summary = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你分析客服对话并总结用户满意度。'},
        {'role':'user','content':f'分析以下客服对话，判断用户是否满意，并给出理由：\n{conversation}'}
    ],
    temperature=0.2)
print(summary.choices[0].message.content)

print('\n反馈信号类型说明：')
print('  显式反馈：点赞/点踩/评分/评论 → 直接但收集率低')
print('  隐式反馈：复制了回答/关闭了页面/感谢/追问 → 间接但收集率高')
print('  最佳实践：两者结合，隐式反馈兜底，显式反馈验证')

### 讨论
- 你平时使用 AI 时会给反馈吗？为什么？
- 如果用户很少主动反馈，怎么获取有效的反馈信号？
- 反馈多了是好事吗？如何避免「反馈疲劳」？

---

## 活动三：综合项目 —— 十节课知识串联

### 活动目标
综合运用前面学到的知识，完成一个小型端到端项目：从输入护栏 → 模型调用 → 输出评估 → 反馈收集。

In [ ]:
# 活动三：综合项目 —— 端到端 AI 客服流水线

print('='*60)
print('综合项目：AI客服流水线')
print('='*60)

# 模拟用户输入
user_query = '我的订单一直没收到，你们是不是骗子公司？我要投诉！'
print(f'用户输入: {user_query}')

# 步骤1: 输入护栏（第10课）
ok, msg = input_guardrail(user_query)
if not ok:
    print(f'[输入护栏] 拦截: {msg}')
else:
    print('[输入护栏] 通过')

# 步骤2: 意图分类 + 角色设定（第5课 - 提示工程）
system_prompt = ('你是一位有同理心的客服专员。'
                '规则：1)先道歉安抚情绪 2)提供具体解决方案 3)语气温暖专业')

# 步骤3: 调用模型（第1-2课）
print('[模型调用] 处理中...')
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':system_prompt},
        {'role':'user','content':user_query}
    ],
    temperature=0.3)  # 低温度，客服场景需要稳定（第2课）
ai_response = r.choices[0].message.content

# 步骤4: 输出护栏（第10课）
warnings = output_guardrail(ai_response)
if warnings:
    print('[输出护栏] 警告:')
    for w in warnings:
        print(f'  - {w}')
else:
    print('[输出护栏] 通过')

# 步骤5: 输出评估（第3-4课）
print('\n[AI评估] 回答质量评估...')
eval_r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是严格的客服质量评估专家。'},
        {'role':'user','content':f'评估以下客服回答的质量（准确性/同理心/解决力各1-5分）：\n{ai_response}'}
    ],
    temperature=0.2)

# 最终输出
print('='*60)
print('最终AI回答：')
print('='*60)
print(ai_response)

print('='*60)
print('质量评估：')
print('='*60)
print(eval_r.choices[0].message.content)

print('这个流水线整合了：')
print('  第1-2课：模型调用与参数控制')
print('  第3-4课：AI评估')
print('  第5课：提示工程（system prompt设计）')
print('  第10课：输入/输出护栏')

---

## 十节课回顾：AI工程化全景

### 知识地图

| 课次 | 主题 | 核心能力 |
|------|------|----------|
| 第1课 | 走进AI工程化 | API调用、角色设定、能力边界 |
| 第2课 | 揭开基础模型的面纱 | Temperature/Top-P、幻觉识别 |
| 第3课 | AI评估入门 | 手工评估、AI做裁判、A/B对比 |
| 第4课 | AI系统评估实践 | 批量评估、模型选择、评估指南 |
| 第5课 | 提示工程 | 提示词迭代、少样本学习、安全 |
| 第6课 | RAG与AI智能体 | RAG系统、语义检索、智能体 |
| 第7课 | 模型微调 | 微调概念、数据准备、方法对比 |
| 第8课 | 数据为王 | 脏数据识别、数据合成、增强 |
| 第9课 | 推理优化 | 速度测量、成本估算、量化 |
| 第10课 | AI架构与进化 | 护栏、反馈、端到端流水线 |

### 核心认知

> AI工程化的核心不是技术本身，而是「如何把技术变成解决问题的手段」。
> 评估比开发更重要。数据比算法更关键。用户反馈比模型参数更珍贵。

### 课后练习
1. 回顾十节课的所有 notebook，选你最有感觉的一个活动重新做一遍
2. 设计一个你自己的 AI 应用方案：场景、模型选择、评估方法、反馈机制
3. 继续学习：推荐资源《AI Engineering》原书，OpenAI Cookbook，LangChain 文档